In [1]:
data_folder="../Data/"

import sys
import os

sys.path.append(os.path.abspath("../Codes"))

sys.path.append(os.path.abspath("../_Deep_Learning"))

from Utils import freq_filter, windowize,  RegionWiseStandardizer

from BCI_Library import read_subject, saveresults_pickle, compute_welch
from BCI_Library import  train_single_model,select_regions_Cohen_effect_size, build_compiled_mlp

2026-04-04 13:25:26.126265: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-04 13:25:26.179490: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-04 13:25:27.715568: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# Powerspectra EEG or MEG convolutional decoder


In [28]:
regioni_per_fold_subjects = {
    1:[[1, 6, 7, 14, 19, 23, 32, 42, 43, 50, 51, 59, 61, 67],
 [1, 6, 7, 14, 19, 23, 26, 31, 32, 42, 43, 50, 59, 61, 67],
 [1, 6, 7, 14, 19, 23, 42, 43, 50, 51, 59, 61, 67],
 [1, 6, 7, 14, 17, 22, 23, 42, 43, 50, 51, 59, 61, 67],
 [1, 6, 7, 14, 23, 26, 43, 50, 51, 59, 61, 67]],

 2:[[4, 14, 32, 33, 44, 46, 47, 48, 49, 50, 62],
 [4, 14, 21, 32, 33, 44, 46, 47, 48, 50, 58, 62],
 [14, 32, 33, 44, 46, 47, 48, 50, 58, 62],
 [4, 14, 32, 33, 44, 46, 47, 48, 50, 58, 62],
 [4, 14, 32, 33, 44, 46, 47, 48, 50, 62]],
 
 4:[[3, 12, 16, 30, 32, 36, 42, 44, 46, 48, 55, 56, 57, 58, 63],
 [2, 3, 4, 12, 16, 30, 32, 44, 46, 47, 48, 56, 58],
 [3, 4, 10, 12, 16, 30, 32, 34, 44, 46, 48, 56, 57, 58],
 [3, 4, 5, 10, 12, 16, 30, 32, 34, 36, 44, 47, 48, 56, 58],
 [2, 3, 4, 12, 16, 30, 32, 34, 41, 44, 46, 48, 56, 58]],
 
 8:[[7, 14, 15, 20, 21, 23, 26, 27, 34, 42, 48, 50, 51, 58],
 [7, 8, 14, 20, 21, 23, 26, 27, 34, 35, 36, 50, 51, 59],
 [0, 7, 14, 15, 17, 21, 23, 26, 27, 34, 47, 50, 51],
 [7, 14, 20, 21, 23, 26, 27, 33, 34, 35, 48, 50, 51, 58],
 [7, 8, 12, 14, 20, 21, 23, 26, 27, 33, 34, 42, 50, 51]],

 18:[[2, 3, 8, 9, 24, 25, 33, 44, 45, 46, 48, 49, 60, 61, 64, 65],
 [2, 3, 21, 24, 25, 32, 33, 39, 44, 45, 46, 48, 49, 60, 61],
 [2, 8, 18, 19, 24, 29, 32, 33, 44, 46, 48, 49, 52, 60, 61, 62],
 [2, 3, 8, 12, 21, 33, 36, 44, 45, 46, 48, 49, 53, 60, 64, 65],
 [2, 3, 8, 9, 12, 18, 19, 25, 33, 39, 44, 46, 48, 60, 61, 65]] 
 }


# sbj 18
# [[2, 3, 8, 9, 24, 25, 33, 44, 45, 46, 48, 49, 60, 61, 64, 65],
#  [2, 3, 21, 24, 25, 32, 33, 39, 44, 45, 46, 48, 49, 60, 61],
#  [2, 8, 18, 19, 24, 29, 32, 33, 44, 46, 48, 49, 52, 60, 61, 62],
#  [2, 3, 8, 12, 21, 33, 36, 44, 45, 46, 48, 49, 53, 60, 64, 65],
#  [2, 3, 8, 9, 12, 18, 19, 25, 33, 39, 44, 46, 48, 60, 61, 65]]


# sbj4
# [[3, 12, 16, 30, 32, 36, 42, 44, 46, 48, 55, 56, 57, 58, 63],
#  [2, 3, 4, 12, 16, 30, 32, 44, 46, 47, 48, 56, 58],
#  [3, 4, 10, 12, 16, 30, 32, 34, 44, 46, 48, 56, 57, 58],
#  [3, 4, 5, 10, 12, 16, 30, 32, 34, 36, 44, 47, 48, 56, 58],
#  [2, 3, 4, 12, 16, 30, 32, 34, 41, 44, 46, 48, 56, 58]]

# sbj 2
# [[4, 14, 32, 33, 44, 46, 47, 48, 49, 50, 62],
#  [4, 14, 21, 32, 33, 44, 46, 47, 48, 50, 58, 62],
#  [14, 32, 33, 44, 46, 47, 48, 50, 58, 62],
#  [4, 14, 32, 33, 44, 46, 47, 48, 50, 58, 62],
#  [4, 14, 32, 33, 44, 46, 47, 48, 50, 62]]

# sbj 1
# [[1, 6, 7, 14, 19, 23, 32, 42, 43, 50, 51, 59, 61, 67],
#  [1, 6, 7, 14, 19, 23, 26, 31, 32, 42, 43, 50, 59, 61, 67],
#  [1, 6, 7, 14, 19, 23, 42, 43, 50, 51, 59, 61, 67],
#  [1, 6, 7, 14, 17, 22, 23, 42, 43, 50, 51, 59, 61, 67],
#  [1, 6, 7, 14, 23, 26, 43, 50, 51, 59, 61, 67]]

# subject 8 
# [[7, 14, 15, 20, 21, 23, 26, 27, 34, 42, 48, 50, 51, 58],
#  [7, 8, 14, 20, 21, 23, 26, 27, 34, 35, 36, 50, 51, 59],
#  [0, 7, 14, 15, 17, 21, 23, 26, 27, 34, 47, 50, 51],
#  [7, 14, 20, 21, 23, 26, 27, 33, 34, 35, 48, 50, 51, 58],
#  [7, 8, 12, 14, 20, 21, 23, 26, 27, 33, 34, 42, 50, 51]]

In [29]:
import os
import sys
import tensorflow.keras as keras
# To be changed
Tutti_soggetti = [4]
DataType = "EEG"
CNN_o_Autoencoder = "Autoencoder" # Autoencoder, # CNN, #SupervisedAutoencoder
model_path_4 = (
"2026-03-05-11_12_10_Encoder_timeseries_Decoder_spectra_log_simpleNormalization"
+ "/")

region_specific = False

model_path_1 = f"../_Deep_Learning/Models_trained/Subject_"
# model path 2 = f"{subject_index}/"
# model path 2.5 = f"{DataType}/"
model_path_3 = f"{CNN_o_Autoencoder}" + ("s/" if CNN_o_Autoencoder=="Autoencoder" else "/")

lib_path = model_path_1 + f"{Tutti_soggetti[0]}/" + f"{DataType}/" + model_path_3 + model_path_4
sys.path.append(os.path.abspath(lib_path))
try:
    del MultiTaskModel_PowerSpectra
    print("ho resettato MultiTaskModel")
except:
    print("non ho resettato MultiTaskModel")

try :
    from _Backup_Library import MultiTaskModel_PowerSpectra
except:
    from Deep_Library_BCI import MultiTaskModel_PowerSpectra
    raise Exception("Non ho caricato dal backup, controlla bene se è quello che vuoi")

use_GRU = True if "GRU" in model_path_4.replace("/","").split("_") else False

if CNN_o_Autoencoder == "CNN":
    model = MultiTaskModel_PowerSpectra(input_shape=(256,1), output_shape=24, use_classifier=True, use_decoder=False)
if CNN_o_Autoencoder == "Autoencoder":
    model = MultiTaskModel_PowerSpectra(input_shape=(256,1), output_shape=24, use_classifier=False, use_decoder=True)
if CNN_o_Autoencoder == "SupervisedAutoencoder":
    try:
        model = MultiTaskModel_PowerSpectra(input_shape=(256,1), output_shape=24, use_classifier=True, use_decoder=True)
    except:
        model = MultiTaskModel_PowerSpectra(input_shape=(256,1), output_shape=24, use_classifier=True, use_decoder=True)


model.build(input_shape=(256,1))

ho resettato MultiTaskModel


## Per fold region selection

In [4]:
def compute_power_spectra(data_2_MI, data_2_Rest):
    sfreq = 250
    miff = [8,12]
    maff = [12,30]

    WMI, WRe, FreqMI = compute_welch(data_2_MI, data_2_Rest, sfreq,
                                        kargs_welch={"fmin":miff[0], "fmax":maff[-1]})

    return WMI, WRe


def selezione_cohen(WRe, WMI, train_idx, num_best_ROIs):
    training_index_Re = train_idx[np.where(train_idx<len(WRe))] 
    training_index_MI = train_idx[np.where(train_idx>=len(WRe))]-len(WRe)

    regions_selected, effect_size  = select_regions_Cohen_effect_size(wpsdMI=WMI[training_index_MI], 
                                                    wpsdRest=WRe[training_index_Re], 
                                                        important_regions=[], nbest_regions=num_best_ROIs)
    return regions_selected

In [30]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold, train_test_split
import tqdm
from Utils import split_create_windows, scale_newax



regioni_per_fold = regioni_per_fold_subjects[Tutti_soggetti[0]]


today = date.today()
sfreq = 250

seed = 224

saveresults = False
filepath_ori = "../Results/BCI_Performances/DNN/BCI_Performances_DNN_subject_"

Features_name = CNN_o_Autoencoder + "_extracted"
Region_Selection= "Cohen's d effect size selection" # "-"

PFR = False

random_seed=224

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

start = 3           # seconds where to start to extract windows
sampling_hz = 250;  start = start*sampling_hz
input_shape = 256   # length of each window
shift = 85          # points to shift for next window in data
num_windows = 7     # how many windows to extract from each trial

end = start + (num_windows-1)*shift + input_shape  # seconds where to end to extract windows (1500 == 6 seconds)


num_best_ROIs = 8


verbose = False

Classifier_names, meths =  ["SVC"],[SVC] # ["MLP"], ["MLP"]
MLP_fit_kwargs={'epochs':20, 'batch_size':32}
DataTypes = [DataType]
rl = None
for DataType in DataTypes:
    for Classifier_name, meth in zip(Classifier_names,meths):
        
        Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Accuracy": [],
            "Features": [],
            "Comments": [],
            "Region Selection": [],
            "Date": []
        })
        if rl is None: rl = Rows
       
        print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
        for subject_index in tqdm.tqdm(Tutti_soggetti):

            model_path = model_path_1 + f"{subject_index}/" + f"{DataType}/" + model_path_3 + model_path_4
            Comments = model_path_4[:-1]+ f" Subject addestramento {subject_index}"

            filepath = filepath_ori + f"{subject_index}.pkl"
            file_input = filepath
            file_output = filepath

            data_2_Rest = read_subject(data_folder, DataType, "Baseline",subject_index,verbose=verbose)
            data_2_MI = read_subject(data_folder, DataType, "MI",subject_index,verbose=verbose)
            data = np.concatenate((data_2_Rest, data_2_MI), axis=0)
            
            WMI, WRe = compute_power_spectra(data_2_MI,data_2_Rest)
            
            data = freq_filter(data, sf=250, freqs=[4,45], type_filter="bandpass") # axis = -1 by default
            data = data[:,:,start:end]        #  data.shape = (192, rois_selected, 748)

            # y shape: (n_trials,) with negative values for Rest and positive for MI
            y = np.concatenate([-np.ones((data_2_Rest.shape[0])), np.ones((data_2_MI.shape[0]))])

            
            fold_accuracies = []
            tutti_fold_selezioni_regioni = []
            for fold_idx, (train_idx, block_idx) in enumerate(kf.split(data, y)):
            ########################## DATA PREPROCESSING ##########################################
            # TODO create (num_windows,0)
                Feat_train_regions_selected = None  #np.empty((len(train_idx)*11,0))
                Feat_test_regions_selected = None # np.empty()
                
                #regions_selected = selezione_cohen(WRe, WMI, train_idx, num_best_ROIs)


                #TODO ECCO QUI IL POSTINO PAT
                regions_selected = regioni_per_fold[fold_idx]
                regions_selected.sort()
                tutti_fold_selezioni_regioni.append(regions_selected)
                
                for region_index, regione in enumerate(regions_selected):
                   
                    split_num = fold_idx+1
                    if region_specific:
                        model.load_weights(model_path+f"Split_{split_num}/Region_{regione}/model.weights.h5")
                    else:
                        model.load_weights(model_path+f"Split_{split_num}/model.weights.h5")
        

                    x_train, y_train, x_val, y_val, x_test, y_test = (
                    split_create_windows(data, y, train_idx, block_idx, 1, win_len=input_shape, shift=shift,
                        random_seed=random_seed, regione=regione)
                        )
                
                    x_train, x_val, x_test = scale_newax (x_train, y_train, x_val, y_val, x_test, y_test, scaler=RegionWiseStandardizer())
                    
                    Features_training_1_regione = model.encoder(x_train)
                    Features_test_1_regione = model.encoder(x_test)
                    # Features_training_1_regione  shape (n_windows,16)
                    
                    
                    ################################### Feature Extractions ##########################################
                    if Feat_train_regions_selected is None:
                        Feat_train_regions_selected = np.empty((Features_training_1_regione.shape[0],0))
                        Feat_test_regions_selected = np.empty((Features_test_1_regione.shape[0],0))

                    Feat_train_regions_selected = np.concatenate((Feat_train_regions_selected,Features_training_1_regione),axis = 1) # asse di regione
                    Feat_test_regions_selected = np.concatenate((Feat_test_regions_selected,Features_test_1_regione),axis = 1) # asse di regione


                Feat_All_regions_selected = np.concatenate((Feat_train_regions_selected,Feat_test_regions_selected))
                # Feat_.shape (Trials, selected_ROIs, features for Roi)
                labels_All = np.concatenate(((y_train > 0).astype(int),(y_test > 0).astype(int)))

                train_idx_model = [_ for _ in range(len(Feat_train_regions_selected))]
                val_idx_model = [_ + len(Feat_train_regions_selected) for _ in range(len(Feat_test_regions_selected))]


                if Classifier_name.lower()=="mlp":
                        meth = build_compiled_mlp(input_dim=Feat_All_regions_selected.shape[1])
                        
                acc, cm, model_classifier = train_single_model(Feat_All_regions_selected, labels_All,
                                                    train_idx_model, val_idx_model, method=meth, seed=224, fit_kwargs=MLP_fit_kwargs)

                fold_accuracies.append(acc)
                if verbose:
                    print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                    print(cm)

                #####################################################################################################################

            if verbose:
                print("\n==================================\n","Mean accuracy:", np.mean(fold_accuracies))
                print("Std accuracy:", np.std(fold_accuracies))

            new_row = {
                "ROIs": [regions_selected],
                "NROIs":[len(regions_selected)],
                "DataType": [DataType],
                "freqs_band": ["-"],  
                "subject": [subject_index],
                "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                "Accuracy": [fold_accuracies],
                "Features": [Features_name],
                "Comments": [Comments],
                "Region Selection": [Region_Selection],
                "Date": [today],
                "Personalized_Freq_Range": ["-"],
            }
            Rows=pd.concat((Rows,pd.DataFrame(new_row)))
            rl = pd.concat((rl,Rows))
        

        if saveresults:
            saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)



##############################
 EEG SVC


100%|██████████| 1/1 [00:15<00:00, 15.24s/it]


In [31]:
rl.Accuracy.apply(np.mean), rl.Classifier, Tutti_soggetti[0]

(0    0.809248
 Name: Accuracy, dtype: float64,
 0    SVC
 Name: Classifier, dtype: object,
 4)